# Auralis Quickstart (Google Colab)

This notebook provisions Auralis on top of **vLLM 0.6.4.post1** with PyTorch 2.5.1 (CUDA 12.1 wheels) and demonstrates the new **streaming** API (`tts.save_stream`) for O(1)-RAM audio generation — safe for audiobook-length texts.

**Requirements:** a Colab runtime with a CUDA GPU (T4 / L4 / A100). On CPU-only runtimes this notebook will error out at import time.

## 1. Install dependencies

Five dedicated cells — ordering matters because numpy / transformers constrain every downstream wheel.

In [ ]:
# Cell 0 — system audio libraries (required by sounddevice / portaudio).
!apt-get install -y -q libportaudio2 libportaudiocpp0 portaudio19-dev

In [ ]:
# Cell 1 — numpy and transformers FIRST (their versions constrain everything else).
!pip install --quiet "numpy<2.0" "transformers==4.44.2"

In [ ]:
# Cell 2 — PyTorch with CUDA 12.1 wheels (matches vLLM 0.6.4 build matrix).
!pip install --quiet torch==2.5.1 torchaudio==2.5.1 \
    --index-url https://download.pytorch.org/whl/cu121

In [ ]:
# Cell 3 — vLLM pinned to the exact version Auralis was validated against.
!pip install --quiet vllm==0.6.4.post1

In [ ]:
# Cell 4 — Auralis itself. --no-deps so pip does not re-resolve torch/vllm/numpy.
# setup.py pulls the rest of the runtime deps (rich, soundfile, langid, cutlet, ...).
!pip install --quiet --force-reinstall --no-deps \
    git+https://github.com/JasperSeling/Auralis.git@main

## 2. Load the XTTSv2 model

In [ ]:
from auralis import TTS, TTSRequest

# gpu_memory_utilization=0.85 — обход известного бага в
# XTTSv2.get_memory_usage_curve(): эмпирический полином возвращает
# "размер весов в ГБ" и передаёт это число в vLLM как долю VRAM, из-за чего
# на T4 16GB получается ~9% → KV-cache остаётся ~250MB → RECOMPUTE-preemption
# на длинных текстах. 0.85 резервирует 85% VRAM под веса+KV-cache
# (на T4 → ~13.6GB, на A100 80GB → ~68GB — с запасом на обеих).
# max_concurrency и scheduler_max_concurrency оставляем в дефолтах (=10),
# чтобы сохранить главное преимущество vLLM-бэкенда — параллельный батчинг
# предложений внутри одного sub-request.
tts = TTS().from_pretrained(
    "AstraMindAI/xttsv2",
    gpt_model="AstraMindAI/xtts2-gpt",
    gpu_memory_utilization=0.85,
)

## 3. Generate speech (streaming, O(1) RAM)

Upload `reference.wav` (a 5-30 sec speaker sample) and `2.txt` (your text) to the Colab file panel, then run the cell below.

`save_stream` writes chunks to disk as they arrive — the full audio is never buffered in memory, so you can generate arbitrarily long texts (whole books). Prefer `.flac` for anything over ~10 min (it is streaming-safe and recovers cleanly from a Colab disconnect).

In [ ]:
import time
from pathlib import Path
from IPython.display import Audio

with open("/content/2.txt", "r", encoding="utf-8") as f:
    text = f.read()
print(f"📖 Текст: {len(text)} символов")

request = TTSRequest(
    text=text,
    language="ru",
    speaker_files=["/content/reference.wav"]
)

OUTPUT_PATH = "/content/output.flac"  # .wav if you need universal compatibility

stats = tts.save_stream(request, OUTPUT_PATH)

print(f"⏱️  Генерация заняла: {stats['wall_sec']:.1f} сек")
print(f"🎵 Длина аудио: {stats['duration_sec']:.1f} сек ({stats['duration_sec']/60:.1f} мин)")
print(f"🚀 Realtime factor: {stats['rtf']:.3f}x")
print(f"💾 Файл: {Path(OUTPUT_PATH).stat().st_size / 1024 / 1024:.1f} MB")

Audio(OUTPUT_PATH)

## 4. Shutdown

Always shut the engine down before the runtime disconnects — this flushes hidden-state temp files and terminates vLLM's background workers cleanly.

In [ ]:
import asyncio
asyncio.get_event_loop().run_until_complete(tts.shutdown())